# exp037 test-time prefix online training audit

## Contents

1. Setup and configuration
2. Load train wells
3. Run prefix online-training audit
4. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, load_config
from test_time_prefix_online_training_audit import run_audit
from pseudo_tail_augmentation import train_files

paths = ExperimentPaths()
config = load_config()
artifacts_dir = paths.artifacts_dir
artifacts_dir.mkdir(parents=True, exist_ok=True)

print('Experiment:', config['experiment']['name'])
print('Route:', config['experiment']['route'])
print('Parent:', config['lineage']['parent'])
print('Control:', config['audit']['control_experiment'])
print('Selected variant:', config['audit']['training_variants']['selected_variant'])
print('Online candidates:', config['audit']['online_training']['candidates'])
print('Online weights:', config['audit']['online_training']['weights'])

## 2. Load train wells

In [ ]:
files = train_files(paths, max_wells=None)
print('Train horizontal wells:', len(files))
print('Train data dir:', paths.train_data_dir)
print('Output artifacts dir:', artifacts_dir)

## 3. Run prefix online-training audit

In [ ]:
summary = run_audit(files=files, config=config, output_dir=artifacts_dir)
print(json.dumps({
    'control_cv': summary['control_cv'],
    'raw_pseudo_tail_cv': summary['raw_pseudo_tail_cv'],
    'best_same_oof_candidate': summary['best_same_oof_candidate'],
    'best_same_oof_cv': summary['best_same_oof_cv'],
    'leave_one_original_fold_out_selection_cv': summary['leave_one_original_fold_out_selection_cv'],
    'well_hash_holdout_selection_cv': summary['well_hash_holdout_selection_cv'],
    'clean_prefix_online_training_supported': summary['clean_prefix_online_training_supported'],
    'selected_method': summary['selected_method'],
    'rules_risk': summary['rules_risk'],
}, indent=2, sort_keys=True))

## 4. Metrics and artifacts

In [ ]:
metrics = pd.read_csv(artifacts_dir / 'prefix_online_training_candidate_metrics.csv')
metrics.sort_values('rmse').head(12)

In [ ]:
bucket_summary = pd.read_csv(artifacts_dir / 'prefix_online_training_bucket_summary.csv')
bucket_summary.sort_values(['bucket', 'rmse']).head(20)

In [ ]:
selection = pd.read_csv(artifacts_dir / 'prefix_online_training_selection.csv')
selection

In [ ]:
online_rows = pd.read_csv(artifacts_dir / 'prefix_online_training_online_rows.csv')
print('Online row summaries:', len(online_rows))
online_rows.head(20)